In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

out = Path('../data/bi')
out.mkdir(parents=True, exist_ok=True)

clean = pd.read_parquet('../data/clean/clean.parquet')

reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
reviews = (reviews.sort_values('review_creation_date')
           .drop_duplicates('order_id', keep='last')[['order_id', 'review_score']])

fact = clean.merge(reviews, on='order_id', how='left')
assert fact['is_late'].sum() == 6534, f"is_late changed: {fact['is_late'].sum()}"

fact['delay_days'] = fact['delivery_days'] - fact['estimated_days']
fact['delay_bucket'] = np.where(fact['is_late'] == 1,
                                np.ceil(fact['delay_days']),
                                np.floor(fact['delay_days'])).astype(int)
fact['purchase_date'] = fact['order_purchase_timestamp'].dt.normalize()

fact = fact[['order_id', 'purchase_date', 'customer_state', 'seller_id',
             'seller_state', 'product_category_name', 'delivery_days',
             'estimated_days', 'is_late', 'delay_days', 'delay_bucket',
             'distance_km', 'review_score', 'total_price', 'total_freight']]

print('fact_orders:', fact.shape)
print(fact.groupby('delay_bucket')['is_late'].mean().loc[-3:3])

fact_orders: (96470, 15)
delay_bucket
-3    0.0
-2    0.0
-1    0.0
 1    1.0
 2    1.0
 3    1.0
Name: is_late, dtype: float64


In [2]:
state_rate = fact.groupby('customer_state')['is_late'].mean()
fact['expected_late'] = fact['customer_state'].map(state_rate)

dim_seller = (fact.groupby('seller_id')
              .agg(seller_state=('seller_state', 'first'),
                   orders=('order_id', 'count'),
                   late_orders=('is_late', 'sum'),
                   expected_late=('expected_late', 'sum'),
                   avg_score=('review_score', 'mean'),
                   revenue=('total_price', 'sum'))
              .reset_index())

dim_seller['excess_late'] = dim_seller['late_orders'] - dim_seller['expected_late']
dim_seller['lift'] = dim_seller['late_orders'] / dim_seller['expected_late']
dim_seller['flagged'] = ((dim_seller['orders'] >= 50) &
                         (dim_seller['lift'] >= 1.5) &
                         (dim_seller['excess_late'] >= 10)).astype(int)

fact.drop(columns='expected_late').to_csv(out / 'fact_orders.csv', index=False)
dim_seller.to_csv(out / 'dim_seller.csv', index=False)

flagged = dim_seller[dim_seller['flagged'] == 1]
print('dim_seller:', dim_seller.shape, '| flagged:', len(flagged))
print('excess:', round(flagged['excess_late'].sum(), 1),
      '| share:', round(flagged['excess_late'].sum() / fact['is_late'].sum() * 100, 1), '%')

dim_seller: (2959, 10) | flagged: 17
excess: 298.3 | share: 4.6 %


In [3]:
dates = pd.date_range(fact['purchase_date'].min(), fact['purchase_date'].max(), freq='D')

dim_date = pd.DataFrame({'date': dates})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%b')
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_of_week'] = dim_date['date'].dt.day_name()

dim_date.to_csv(out / 'dim_date.csv', index=False)
print('dim_date:', dim_date.shape, dim_date['date'].min().date(), '-', dim_date['date'].max().date())

dim_date: (714, 7) 2016-09-15 - 2018-08-29
